# Medical Coding Assistant — RAG + Thaura API

**Pipeline overview:**
1. Install dependencies
2. Load API key (safely)
3. Extract data from BigQuery (ETL)
4. Build vector knowledge base (RAG)
5. Predict ICD-10 codes using GPT-3.5-turbo
6. Evaluate accuracy on validation set


## Install Dependencies

In [ ]:
!pip install -q sentence-transformers

In [ ]:
!pip install -q langchain-huggingface langchain-community langchain-core
!pip install -q chromadb
!pip install -q openai
!pip install -q transformers torch accelerate

##Carbontracker

In [ ]:
!pip install carbontracker

In [ ]:
import glob, os, re, shutil
from carbontracker.tracker import CarbonTracker

def parse_carbon_log(log_dir):
    US_CARBON_INTENSITY = 383.55

    log_files = [f for f in glob.glob(f'{log_dir}/*.log')
                 if 'output' not in f and 'err' not in f]

    if not log_files:
        print(f'  ⚠ No log found in {log_dir}')
        return None, None

    for log_path in sorted(log_files, reverse=True):
        with open(log_path, 'r') as f:
            content = f.read()

        # try official consumption line first
        energy_match = re.search(r'Actual consumption.*?Energy:\s*(\d+\.?\d+)\s*kWh',
                                  content, re.IGNORECASE | re.DOTALL)
        if energy_match:
            energy_kwh = float(energy_match.group(1))
            co2_grams  = round(energy_kwh * US_CARBON_INTENSITY, 6)
            return co2_grams, energy_kwh

        # fallback: calculate from duration + power
        duration_match = re.search(r'Duration:\s*(\d+):(\d+):(\d+)', content)
        power_match    = re.search(r'Average power usage \(W\) for cpu:\s*([\d.]+)', content)

        if duration_match and power_match:
            hours      = (int(duration_match.group(1)) +
                          int(duration_match.group(2)) / 60 +
                          float(duration_match.group(3)) / 3600)
            power_w    = float(power_match.group(1))
            energy_kwh = round((power_w * hours) / 1000, 9)
            co2_grams  = round(energy_kwh * US_CARBON_INTENSITY, 6)
            return co2_grams, energy_kwh

    print(f'  ⚠ No data found in any log in {log_dir}')
    return None, None

# ── helper: run a stage with its own tracker ─────────────────────────────────

def run_with_carbon(stage_name, fn, notebook, output_path):
    """
    Wraps any function call with CarbonTracker.
    Saves the log to its own subdirectory so it's never overwritten.

    Usage:
        result = run_with_carbon('rag_eval', lambda: my_function(), 'single', OUTPUT_PATH)
    """
    log_dir = f'{output_path}/carbon_logs/{stage_name}'
    os.makedirs(log_dir, exist_ok=True)

    tracker = CarbonTracker(epochs=1, log_dir=log_dir, monitor_epochs=1, verbose=0)
    tracker.epoch_start()
    result = fn()
    tracker.epoch_end()
    tracker.stop()

    co2, energy = parse_carbon_log(log_dir)
    print(f'  ✓ [{stage_name}] CO2: {co2:.6f} g | Energy: {energy} kWh' if co2 else
          f'  ⚠ [{stage_name}] Could not parse carbon log')

    return result, {'stage': stage_name, 'notebook': notebook,
                    'co2_grams': co2, 'energy_kwh': energy}


## Load API Key

**How to store your Thaura key:**
1. Go to **thaura.ai** → API Keys → Create new key
2. In Colab, click the key icon in the left sidebar
3. Add a secret named `THAURA_API_KEY` and paste your key

**Never paste your key directly into a code cell.**

In [ ]:
# cost tracker (relevant later on)
token_tracker = {
    'rag':    {'input': 0, 'output': 0, 'calls': 0},
    'no_rag': {'input': 0, 'output': 0, 'calls': 0}
}

In [ ]:
from google.colab import userdata
from openai import OpenAI

# load key from Colab secrets panel
THAURA_API_KEY = userdata.get('THAURA_API_KEY')

# initialize the client
thaura_client = OpenAI(api_key=THAURA_API_KEY, base_url="https://backend.thaura.ai/v1")

# make sure the key works before doing anything else
test_response = thaura_client.chat.completions.create(
    model='thaura',
    messages=[{'role': 'user', 'content': 'Say "API key works!" and nothing else.'}],
    max_tokens=2000
)
print(test_response.choices[0].message.content)


API key works!


## ETL: Extract Data from BigQuery

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from google.cloud import bigquery
from google.colab import auth
import pandas as pd
import os

auth.authenticate_user()

OUTPUT_PATH = '/content/drive/MyDrive/mimic4_medical_coding'
os.makedirs(OUTPUT_PATH, exist_ok=True)

client = bigquery.Client(project='perfect-crow-472910-d7')
print('✓ Connected to BigQuery')

Mounted at /content/drive
✓ Connected to BigQuery


## Build the RAG Knowledge Base

This embeds ICD-10 guidelines and training examples into a vector database.
When predicting, we search this DB for similar documents and pass them
as context to GPT.

**Note:** If you already ran this once today, skip creating synthetic notes since they are saved to Drive and you can just reload them.

In [ ]:
# ETL (Extract, Transform, Load): Generate synthetic notes + pull ICD guidelines from BigQuery

from google.cloud import bigquery
from google.colab import auth
import pandas as pd
import os, json, re

auth.authenticate_user()

OUTPUT_PATH = '/content/drive/MyDrive/mimic4_medical_coding'
os.makedirs(OUTPUT_PATH, exist_ok=True)

bq_client = bigquery.Client(project='perfect-crow-472910-d7')
print('✓ Connected to BigQuery')


# pull ICD-10 guidelines from BigQuery
def extract_icd10_guidelines(client):
    query = """
    SELECT DISTINCT icd_code, long_title as description
    FROM `physionet-data.mimiciv_3_1_hosp.d_icd_diagnoses`
    WHERE icd_version = 10
    ORDER BY icd_code
    """
    df = client.query(query).to_dataframe()
    print(f'✓ Extracted {len(df)} ICD-10 guidelines')
    return df


# generate synthetic clinical notes using GPT
def generate_synthetic_notes():
    codes_to_generate = [
        # cardiac
        ('I214',  'ST elevation myocardial infarction'),
        ('I480',  'Paroxysmal atrial fibrillation'),
        ('I10',   'Essential hypertension'),
        ('I500',  'Congestive heart failure'),
        ('I209',  'Angina pectoris, unspecified'),
        # respiratory
        ('J189',  'Pneumonia, unspecified'),
        ('J449',  'Chronic obstructive pulmonary disease, unspecified'),
        ('J960',  'Acute respiratory failure'),
        # metabolic
        ('E119',  'Type 2 diabetes without complications'),
        ('E1165', 'Type 2 diabetes with hyperglycemia'),
        ('E876',  'Hypokalemia'),
        ('E860',  'Hypovolemia'),
        # renal
        ('N179',  'Acute kidney failure, unspecified'),
        ('N189',  'Chronic kidney disease, unspecified'),
        # infectious
        ('A419',  'Sepsis, unspecified organism'),
        ('B9721', 'Sepsis due to MRSA'),
        # neurological
        ('R55',   'Syncope and collapse'),
        ('G4733', 'Obstructive sleep apnea'),
        ('I639',  'Cerebral infarction, unspecified'),
        # gastrointestinal
        ('K921',  'Melena'),
        ('K8020', 'Calculus of gallbladder without cholecystitis'),
        # oncology
        ('C250',  'Malignant neoplasm of head of pancreas'),
        ('C3490', 'Malignant neoplasm of bronchus and lung'),
        # other
        ('S0690', 'Unspecified intracranial injury'),
        ('T8401', 'Breakdown of cardiac electrode'),
    ]

    BATCHES       = 3       # API calls per code
    NOTES_PER_BATCH = 5    # notes per call

    synthetic_notes = []

    for code, description in codes_to_generate:
        code_notes = []
        print(f'Generating notes for {code} ({description})...', end=' ')

        for batch in range(BATCHES):
            prompt = f"""Generate {NOTES_PER_BATCH} fictional discharge notes where the PRIMARY diagnosis
is SPECIFICALLY {description} (ICD-10: {code}).

STRICT RULES:
- The diagnosis MUST be {description} — do NOT describe a more specific or related condition
- For example, if the code is NSTEMI, do NOT mention ST elevation anywhere in the note
- If the code is 'unspecified', the note must NOT contain findings that point to a specific subtype
- Each note must be 150-250 words
- Include: chief complaint, history, physical exam, hospital course, discharge plan
- Use medical abbreviations and clinical language
- NO real names, dates, or identifying information
- Do NOT mention any ICD-10 codes, diagnosis codes, or code numbers in the note
- Do NOT write out the full official diagnosis name verbatim — describe it clinically

SELF-CHECK: Before returning, verify each note's symptoms match ONLY {description} and not a related condition.

Return ONLY a JSON array:
[
  {{"note": "Chief complaint: ...", "code": "{code}"}},
  {{"note": "Chief complaint: ...", "code": "{code}"}}
]"""
            try:
                response = thaura_client.chat.completions.create(
                    model='thaura',
                    messages=[
                        {'role': 'system', 'content': 'You are a medical education content creator generating fictional case studies.'},
                        {'role': 'user',   'content': prompt}
                    ],
                    max_tokens=8000,
                    temperature=0.8
                )
                raw   = response.choices[0].message.content.strip()
                clean = re.sub(r'```json|```', '', raw).strip()
                notes = json.loads(clean)
                code_notes.extend(notes)

            except Exception as e:
                print(f'\n  ⚠ Batch {batch+1} error for {code}: {e}')

        synthetic_notes.extend(code_notes)
        print(f'✓ {len(code_notes)} notes')

    df = pd.DataFrame(synthetic_notes).rename(columns={'note': 'discharge_summary', 'code': 'primary_icd_code'})
    print(f'\nTotal synthetic notes generated: {len(df)}')
    return df


# split into train/val/test
def create_train_val_test_split(df, train_ratio=0.7, val_ratio=0.15, min_samples=6):
    from sklearn.model_selection import train_test_split

    code_counts = df['primary_icd_code'].value_counts()
    valid_codes = code_counts[code_counts >= min_samples].index

    df_common = df[df['primary_icd_code'].isin(valid_codes)].copy()
    df_rare   = df[~df['primary_icd_code'].isin(valid_codes)].copy()

    df_common['_strat'] = df_common['primary_icd_code'].fillna('UNKNOWN')

    train_df, temp_df = train_test_split(
        df_common, test_size=(1 - train_ratio),
        stratify=df_common['_strat'], random_state=42
    )
    val_size = val_ratio / (1 - train_ratio)
    val_df, test_df = train_test_split(
        temp_df, test_size=(1 - val_size),
        stratify=temp_df['_strat'], random_state=42
    )

    train_df = pd.concat([train_df, df_rare], ignore_index=True)

    for split in [train_df, val_df, test_df]:
        split.drop(columns=['_strat'], inplace=True, errors='ignore')

    print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
    return train_df, val_df, test_df


# --- RUN ETL ---
guidelines_df = extract_icd10_guidelines(bq_client)
synthetic_df  = generate_synthetic_notes()

# filter out any codes not in official guidelines
valid_icd    = set(guidelines_df['icd_code'])
synthetic_df = synthetic_df[synthetic_df['primary_icd_code'].isin(valid_icd)]
print(f'✓ Filtered to {len(synthetic_df)} notes with valid ICD-10 codes')

synthetic_df['primary_icd_description'] = synthetic_df['primary_icd_code'].map(
    dict(zip(guidelines_df['icd_code'], guidelines_df['description']))
)
synthetic_df['hadm_id'] = range(len(synthetic_df))

train_df, val_df, test_df = create_train_val_test_split(synthetic_df)

# save to Drive
guidelines_df.to_csv(f'{OUTPUT_PATH}/icd10_guidelines.csv', index=False)
synthetic_df.to_csv(f'{OUTPUT_PATH}/synthetic_notes.csv',   index=False)
train_df.to_csv(f'{OUTPUT_PATH}/train.csv',                 index=False)
val_df.to_csv(f'{OUTPUT_PATH}/val.csv',                     index=False)
test_df.to_csv(f'{OUTPUT_PATH}/test.csv',                   index=False)
print('✓ All files saved to Drive')

✓ Connected to BigQuery
✓ Extracted 97441 ICD-10 guidelines
Generating notes for I214 (ST elevation myocardial infarction)... ✓ 15 notes
Generating notes for I480 (Paroxysmal atrial fibrillation)... ✓ 15 notes
Generating notes for I10 (Essential hypertension)... ✓ 15 notes
Generating notes for I500 (Congestive heart failure)... ✓ 15 notes
Generating notes for I209 (Angina pectoris, unspecified)... ✓ 15 notes
Generating notes for J189 (Pneumonia, unspecified)... ✓ 15 notes
Generating notes for J449 (Chronic obstructive pulmonary disease, unspecified)... ✓ 15 notes
Generating notes for J960 (Acute respiratory failure)... ✓ 15 notes
Generating notes for E119 (Type 2 diabetes without complications)... ✓ 15 notes
Generating notes for E1165 (Type 2 diabetes with hyperglycemia)... ✓ 15 notes
Generating notes for E876 (Hypokalemia)... ✓ 15 notes
Generating notes for E860 (Hypovolemia)... ✓ 15 notes
Generating notes for N179 (Acute kidney failure, unspecified)... ✓ 15 notes
Generating notes for

In [ ]:
import warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('sentence_transformers').setLevel(logging.ERROR)
logging.getLogger('chromadb').setLevel(logging.ERROR)

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# Bio_ClinicalBERT is trained on clinical text so it understands
# medical language much better than a general embedding model
print('Loading Bio_ClinicalBERT embeddings...')
embeddings = HuggingFaceEmbeddings(
    model_name='emilyalsentzer/Bio_ClinicalBERT',
    model_kwargs={'device': 'cpu'}
)
print('✓ Embeddings model loaded')

In [ ]:
# clinical keywords for common codes — add as many as are relevant to your 25 codes
def build_knowledge_base(train_df, guidelines_df, embeddings,
                         examples_per_code=7):

    # always start fresh
    chroma_path = f'{OUTPUT_PATH}/chroma_db'
    if os.path.exists(chroma_path):
        shutil.rmtree(chroma_path)
        print('✓ Cleared old vector store')

    documents = []
    top_codes = train_df['primary_icd_code'].value_counts().index

    for code in top_codes:
        code_examples = train_df[train_df['primary_icd_code'] == code].head(examples_per_code)

        for _, row in code_examples.iterrows():
            summary = str(row['discharge_summary'])[:1200] if pd.notna(row['discharge_summary']) else ''
            # include the code AND description in the document so
            # the embedding captures both clinical text and label
            desc = guidelines_df[guidelines_df['icd_code'] == code]['description'].values
            desc = desc[0] if len(desc) > 0 else ''

            documents.append(Document(
                page_content=(
                    f"Diagnosis: {desc}\n"
                    f"Clinical Note:\n{summary}"
                ),
                metadata={'type': 'example', 'code': code}
            ))

    print(f'  Total documents: {len(documents)}')
    print('Embedding documents...')

    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        persist_directory=f'{OUTPUT_PATH}/chroma_db'
    )
    print('✓ Knowledge base ready!')
    return vectorstore

def load_existing_knowledge_base(embeddings):
    """Skip rebuilding if you already ran this today — just reload from disk."""
    vectorstore = Chroma(
        persist_directory=f'{OUTPUT_PATH}/chroma_db',
        embedding_function=embeddings
    )
    print(f'✓ Loaded existing knowledge base from disk')
    return vectorstore


# Set REBUILD = False after the first run to save time
'''REBUILD = TRUE

if REBUILD:
    vectorstore = build_knowledge_base(train_df, guidelines_df, embeddings)
else:
    vectorstore = load_existing_knowledge_base(embeddings)'''

'REBUILD = TRUE\n\nif REBUILD:\n    vectorstore = build_knowledge_base(train_df, guidelines_df, embeddings)\nelse:\n    vectorstore = load_existing_knowledge_base(embeddings)'

In [ ]:
# load synthetic notes
OUTPUT_PATH = '/content/drive/MyDrive/mimic4_medical_coding'

guidelines_df = pd.read_csv(f'{OUTPUT_PATH}/icd10_guidelines.csv')
synthetic_df  = pd.read_csv(f'{OUTPUT_PATH}/synthetic_notes.csv')
train_df      = pd.read_csv(f'{OUTPUT_PATH}/train.csv')
val_df        = pd.read_csv(f'{OUTPUT_PATH}/val.csv')
test_df       = pd.read_csv(f'{OUTPUT_PATH}/test.csv')

print(f'✓ guidelines : {len(guidelines_df):,} rows')
print(f'✓ synthetic  : {len(synthetic_df):,} rows')
print(f'✓ train      : {len(train_df):,} rows')
print(f'✓ val        : {len(val_df):,} rows')
print(f'✓ test       : {len(test_df):,} rows')

✓ guidelines : 97,441 rows
✓ synthetic  : 345 rows
✓ train      : 241 rows
✓ val        : 51 rows
✓ test       : 53 rows


In [ ]:
#if you are rebuilding
from carbontracker.tracker import CarbonTracker

vectorstore, carbon_build = run_with_carbon(
    'vector_store_build',
    lambda: build_knowledge_base(train_df, guidelines_df, embeddings),
    notebook='single',
    output_path=OUTPUT_PATH
)

CarbonTracker: INFO - Detected CPU: Intel(R) Xeon(R) CPU @ 2.20GHz
CarbonTracker: WARNING - No matching TDP found for CPU: Intel(R) Xeon(R) CPU @ 2.20GHz. Using average TDP of 35.61W at 50% utilization as fallback.
CarbonTracker: WARNING - No API keys provided. Skipping intensity provider initialization.
CarbonTracker: WARNING - No carbon intensity provider specified. Using average carbon intensity for US: 383.55 gCO2eq/kWh.
  Total documents: 161
Embedding documents...
✓ Knowledge base ready!
  ✓ [vector_store_build] CO2: 0.698128 g | Energy: 0.001820174 kWh


## Predict ICD-10 Codes with RAG + GPT

**How a single prediction works:**
```
Patient note
    │
    ▼
Vector search → finds similar guidelines + past cases
    │
    ▼
Prompt = note + retrieved context (from RAG part)
    │
    ▼
GPT-3.5-turbo generates ICD-10 codes grounded in that context
    │
    ▼  (only if GPT fails/returns bad JSON)
Rule-based fallback → ranks codes by retrieval score
```

In [ ]:
import json, re
from typing import List, Tuple, Dict


def retrieve_context(vectorstore, discharge_summary: str, k: int = 10) -> dict:
    docs = vectorstore.similarity_search(discharge_summary, k=20)

    # rerank
    if docs:
        enriched = f"{discharge_summary}\n\nLooking for ICD-10 codes related to the primary diagnosis."
        pairs  = [(enriched, doc.page_content) for doc in docs]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
        docs   = [doc for _, doc in ranked[:k]]

    # deduplicate by code — keep best-scoring doc per code
    seen  = set()
    deduped = []
    for doc in docs:
        code = doc.metadata['code']
        if code not in seen:
            seen.add(code)
            deduped.append(doc)

    return {'examples': deduped, 'all': deduped}

def build_prompt(discharge_summary, context_docs):
    candidate_codes = list(dict.fromkeys(
        [d.metadata['code'] for d in context_docs]
    ))

    example_text = '\n'.join([
        f"- Code {d.metadata['code']}: "
        f"{d.page_content[d.page_content.find('Clinical Note:')+14:][:150].strip()}"
        for d in context_docs[:5]
    ])

    prompt = f"""You are an expert medical coder. Assign ICD-10 codes to the clinical note below.
You MUST only choose codes from this list: {candidate_codes}

SIMILAR PAST CASES:
{example_text}

CLINICAL NOTE:
{discharge_summary[:500]}

Output ONLY this JSON with exactly 3 codes, nothing else:
{{"codes":[
  {{"code":"A","confidence":0.9,"reason":"brief"}},
  {{"code":"B","confidence":0.8,"reason":"brief"}},
  {{"code":"C","confidence":0.7,"reason":"brief"}}
]}}"""

    return prompt, candidate_codes

from sentence_transformers import CrossEncoder

# This re-ranker scores each (query, document) pair together rather than
# separately — much more accurate than cosine similarity alone.
# MiniLM is small enough to run on CPU in Colab without slowing things down.
print('Loading cross-encoder re-ranker...')
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print('✓ Re-ranker loaded')

def extract_json_from_text(text):
    """Find and extract the JSON codes block, ignoring anything after it closes."""

    # find the opening of the codes object
    start = text.find('{"codes"')
    if start == -1:
        start = text.find('{ "codes"')
    if start == -1:
        # last resort: find first {
        start = text.find('{')

    if start == -1:
        return None

    # walk forward counting braces to find the EXACT closing }
    depth = 0
    in_string = False
    escape_next = False

    for i, ch in enumerate(text[start:], start):
        if escape_next:
            escape_next = False
            continue
        if ch == '\\' and in_string:
            escape_next = True
            continue
        if ch == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                return text[start:i+1]  # stop exactly here, ignore trailing text

    return None

import time

def call_api_with_retry(messages, max_tokens=1500, max_retries=3):
    """
    Retries the API call up to max_retries times with exponential backoff.
    Returns (extracted_json, usage) or raises after all retries exhausted.
    """
    last_error = None

    for attempt in range(max_retries):
        if attempt > 0:
            wait = 2 ** attempt  # 2s, 4s, 8s
            print(f'  Retry {attempt}/{max_retries} after {wait}s...')
            time.sleep(wait)

        try:
            response = thaura_client.chat.completions.create(
                model='thaura',
                messages=messages,
                max_tokens=max_tokens,
                temperature=0.1
            )
            msg = response.choices[0].message
            reasoning = getattr(msg, 'reasoning', None) or ''
            content   = msg.content or ''

            # search content first, then reasoning, then both combined
            for source in [content, reasoning, content + '\n' + reasoning]:
                text = source.strip()
                if not text:
                    continue
                extracted = extract_json_from_text(text)
                if extracted:
                    return extracted, response.usage

            raise ValueError(
                f'No JSON found. finish={response.choices[0].finish_reason} '
                f'content_start={content[:80]!r}'
            )

        except Exception as e:
            last_error = e
            print(f'  Attempt {attempt+1} failed: {e}')
            continue

    raise RuntimeError(f'All {max_retries} attempts failed. Last error: {last_error}')

def predict_codes(discharge_summary, vectorstore, top_k=3, return_context=False):
    context         = retrieve_context(vectorstore, discharge_summary)
    predictions     = []
    used_llm        = False
    candidate_codes = []

    def clean_and_parse(raw_text, candidate_codes, top_k):
        clean = re.sub(r'```json|```', '', raw_text).strip()
        clean = re.sub(r'//[^\n]*', '', clean)
        clean = re.sub(r',\s*([}\]])', r'\1', clean)
        clean = clean.replace("'", '"')
        first_open = clean.find('{')
        if first_open != -1:
            clean = clean[first_open:]
        last_close = clean.rfind('}')
        if last_close != -1:
            clean = clean[:last_close + 1]
            clean += ']' * (clean.count('[') - clean.count(']'))
            clean += '}' * (clean.count('{') - clean.count('}'))
        clean = re.sub(r',\s*\{\s*"code"[^}]*$', '', clean)
        clean = re.sub(r',\s*([}\]])', r'\1', clean)

        try:
            data = json.loads(clean)
        except json.JSONDecodeError:
            raw_codes = re.findall(r'"code"\s*:\s*"([^"]+)"', raw_text)
            return [
                (c.replace('.', ''), 0.5, 'regex extracted')
                for c in raw_codes[:top_k]
                if c.replace('.', '') in set(candidate_codes)
            ]

        codes = data.get('codes', [])
        filtered = [
            (c['code'].replace('.', ''), c.get('confidence', 0.5), c.get('reason', ''))
            for c in codes[:top_k]
            if c['code'].replace('.', '') in set(candidate_codes)
        ]

        seen = {p[0] for p in filtered}
        for code in candidate_codes:
            if len(filtered) >= top_k:
                break
            if code not in seen:
                filtered.append((code, 0.1, 'padded'))
                seen.add(code)

        return filtered

    try:
        # updated to new single-argument signature
        prompt, candidate_codes = build_prompt(
            discharge_summary, context['examples']
        )
        messages = [
            {
                'role': 'system',
                'content': (
                    'You are a JSON API. Output ONLY a raw JSON object. '
                    'No reasoning, no explanation, no prose. '
                    'Your entire response must be valid JSON starting with { and ending with }. '
                    'Maximum 150 tokens.'
                )
            },
            {'role': 'user', 'content': prompt}
        ]

        raw_text, usage = call_api_with_retry(messages)
        predictions = clean_and_parse(raw_text, candidate_codes, top_k)
        used_llm    = True

        token_tracker['rag']['input']  += usage.prompt_tokens
        token_tracker['rag']['output'] += usage.completion_tokens
        token_tracker['rag']['calls']  += 1

    except Exception as e:
        print(f'  All retries failed: {e}')

    result = {'predictions': predictions, 'used_llm': used_llm}

    if return_context:
        result['context'] = {
            # no more guidelines key — just examples
            'examples': [d.page_content for d in context['examples'][:5]]
        }

    return result

## Single Prediction Test

Always run this before the full eval to make sure everything is working.

In [ ]:
def remove_diagnoses_section(note):
    if pd.isna(note):
        return None
    cutoff = note.find('DISCHARGE DIAGNOSES')
    if cutoff != -1:
        note = note[:cutoff].strip()
    # also remove any inline ICD code patterns like (C250) or C250
    note = re.sub(r'\b[A-Z]\d{2,4}\w*\b', '', note)
    note = re.sub(r'\([A-Z]\d{2,4}\w*\)', '', note)
    return note.strip()

In [ ]:
# Test on first row that actually has a note
valid_rows = val_df[val_df['discharge_summary'].notna() & (val_df['discharge_summary'] != '')]
test_row = valid_rows.iloc[0]
test_note = test_row['discharge_summary']
true_code = test_row['primary_icd_code']
true_desc = test_row['primary_icd_description']

print('=' * 60)
print('SINGLE PREDICTION TEST')
print('=' * 60)
print(f'True Code:   {true_code}')
print(f'Description: {true_desc}')
print(f'\nNote (first 400 chars):')
print(test_note[:400])
print('...')

test_note_masked = remove_diagnoses_section(test_note)
result = predict_codes(test_note_masked, vectorstore, top_k=3, return_context=True)

print('\n' + '=' * 60)
print(f"Method: {'Thaura' if result['used_llm'] else 'Rule-based fallback'}")
print('=' * 60)
print('\nPREDICTIONS:')
for i, (code, conf, reason) in enumerate(result['predictions'], 1):
    match = ' (true) ' if code in true_code else ' (false) '
    print(f'  {i}. {code}  ({conf:.0%} confidence)  {reason[:60]}{match}')

print('\nCONTEXT RETRIEVED (what the model saw):')
for e in result['context']['examples']:
    print(f'    · {e[:200]}')   # longer slice so it's readable
    print()                     # blank line between docs

SINGLE PREDICTION TEST
True Code:   E119
Description: Type 2 diabetes mellitus without complications

Note (first 400 chars):
Chief complaint: Frequent urination and increased thirst for 2 weeks
History: 67-year-old female with history of obesity presents with polyuria and polydipsia for 2 weeks. No weight change, visual disturbances, or extremity symptoms. Reports occasional fatigue but denies chest pain or shortness of breath. Takes no medications regularly. Family history non-contributory.
Physical exam: BP 138/86. BM
...

Method: Thaura

PREDICTIONS:
  1. E119  (90% confidence)  Polyuria/polydipsia with obesity (true) 
  2. E1165  (70% confidence)  Diabetes symptoms but no history (false) 
  3. I10  (50% confidence)  No hypertension mentioned (false) 

CONTEXT RETRIEVED (what the model saw):
    · Diagnosis: Type 2 diabetes mellitus with hyperglycemia
Clinical Note:
Chief complaint: Polyuria, polydipsia, and fatigue for the past 2 weeks. History: Patient is a 58-year-old male with 

## Evaluate on Validation Set


In [ ]:
import os
import json
import pandas as pd

import shutil
for stage in ['rag_eval', 'hallucination_eval', 'vector_store_build']:
    log_dir = f'{OUTPUT_PATH}/carbon_logs/{stage}'
    if os.path.exists(log_dir):
        shutil.rmtree(log_dir)
    os.makedirs(log_dir)
print('Carbon log directories cleared')

EVAL_SAMPLES = 50
carbon_log_dir = f'{OUTPUT_PATH}/carbon_logs'
os.makedirs(carbon_log_dir, exist_ok=True)


print(f'Evaluating {EVAL_SAMPLES} validation samples with carbon tracking...')
print('Each dot = incorrect (corrrect not in top-3)| ✓ = correct in top-3\n')

val_df_clean = val_df[
    val_df['discharge_summary'].notna() & (val_df['discharge_summary'] != '')
].copy()

def run_eval():
    local_results = []
    for idx, row in val_df_clean.head(EVAL_SAMPLES).iterrows():
        note = remove_diagnoses_section(row['discharge_summary'])
        result = predict_codes(note, vectorstore, top_k=3)

        true_code       = row['primary_icd_code']
        predicted_codes = [p[0] for p in result['predictions']]

        # handle empty predictions gracefully
        top1 = predicted_codes[0] == true_code if predicted_codes else False
        top3 = true_code in predicted_codes if predicted_codes else False

        local_results.append({
            'hadm_id':           row['hadm_id'],
            'true_code':         true_code,
            'predicted_codes':   predicted_codes,
            'confidence_scores': [p[1] for p in result['predictions']],
            'top1_correct':      top1,
            'top3_correct':      top3,
            'used_llm':          result['used_llm']
        })

        print('✓' if top3 else '.', end='', flush=True)
    return local_results

results, carbon_eval = run_with_carbon(
    'rag_eval',
    run_eval,
    notebook='single',
    output_path=OUTPUT_PATH
)

results_df = pd.DataFrame(results)
results_df.to_csv(f'{OUTPUT_PATH}/single_results.csv', index=False)
print('\n')
print(f"Top-1 Accuracy : {results_df['top1_correct'].mean():.1%}")
print(f"Top-3 Accuracy : {results_df['top3_correct'].mean():.1%}")

Carbon log directories cleared
Evaluating 50 validation samples with carbon tracking...
Each dot = incorrect (corrrect not in top-3)| ✓ = correct in top-3

CarbonTracker: INFO - Detected CPU: Intel(R) Xeon(R) CPU @ 2.20GHz
CarbonTracker: WARNING - No matching TDP found for CPU: Intel(R) Xeon(R) CPU @ 2.20GHz. Using average TDP of 35.61W at 50% utilization as fallback.
CarbonTracker: WARNING - No API keys provided. Skipping intensity provider initialization.
CarbonTracker: WARNING - No carbon intensity provider specified. Using average carbon intensity for US: 383.55 gCO2eq/kWh.
✓✓✓✓✓✓✓✓  Attempt 1 failed: No JSON found. finish=length content_start=''
  Retry 1/3 after 2s...
✓✓.✓✓✓  Attempt 1 failed: No JSON found. finish=length content_start=''
  Retry 1/3 after 2s...
✓✓✓✓✓✓✓✓✓✓  Attempt 1 failed: No JSON found. finish=length content_start=''
  Retry 1/3 after 2s...
  Attempt 2 failed: No JSON found. finish=length content_start=''
  Retry 2/3 after 4s...
✓✓✓✓✓✓✓✓✓✓✓✓✓  Attempt 1 failed

In [ ]:
code_to_desc = dict(zip(guidelines_df['icd_code'], guidelines_df['description']))

# ── CODE ACCURACY BREAKDOWN ──────────────────────────────────────────────────
# one true code per note

code_accuracy = results_df.groupby('true_code').agg(
    total     = ('top1_correct', 'count'),
    top1_hits = ('top1_correct', 'sum'),
    top3_hits = ('top3_correct', 'sum'),
).reset_index()

code_accuracy['top1_accuracy'] = code_accuracy['top1_hits'] / code_accuracy['total']
code_accuracy['top3_accuracy'] = code_accuracy['top3_hits'] / code_accuracy['total']
code_accuracy['icd_chapter']   = code_accuracy['true_code'].str[0]
code_accuracy['description']   = code_accuracy['true_code'].map(code_to_desc).fillna('Unknown')

code_accuracy.to_csv(f'{OUTPUT_PATH}/code_accuracy_breakdown.csv', index=False)

print('CODE ACCURACY BREAKDOWN saved.')
print(f"  {len(code_accuracy)} unique codes evaluated")
print(f"  Overall top-1: {results_df['top1_correct'].mean():.1%}")
print(f"  Overall top-3: {results_df['top3_correct'].mean():.1%}")
print()

# ── CONFIDENCE DISTRIBUTION ──────────────────────────────────────────────────
# Explode predictions so each predicted code is its own row.
# rank=1 is the model's top pick, rank=5 is its last.

conf_rows = []
for _, r in results_df.iterrows():
    preds       = r['predicted_codes']    # list of codes
    confidences = r['confidence_scores']  # list of floats (same length)

    for rank, (code, conf) in enumerate(zip(preds, confidences)):
        conf_rows.append({
            'hadm_id':        r['hadm_id'],
            'true_code':      r['true_code'],
            'predicted_code': code,
            'rank':           rank + 1,                    # 1 = model's top pick
            'confidence':     round(conf, 4),
            'is_correct':     int(code == r['true_code']), # did this prediction match the one true code?
            'note_top1':      int(r['top1_correct']),
            'note_top3':      int(r['top3_correct']),
            'icd_chapter':    code[0] if code else '',
        })

confidence_df = pd.DataFrame(conf_rows)
confidence_df.to_csv(f'{OUTPUT_PATH}/confidence_distribution.csv', index=False)

print('CONFIDENCE DISTRIBUTION saved.')
print(f"  {len(confidence_df)} predicted-code rows ({len(results_df)} notes × avg {len(confidence_df)/len(results_df):.1f} predictions each)")
print(f"  Correct predictions : {confidence_df['is_correct'].sum()} ({confidence_df['is_correct'].mean():.1%})")
print()
print('Avg confidence by correctness:')
print(confidence_df.groupby('is_correct')['confidence'].mean().round(3).rename({0: 'Wrong', 1: 'Correct'}).to_string())
print()
print('Avg confidence by rank:')
print(confidence_df.groupby('rank')['confidence'].mean().round(3).to_string())


CODE ACCURACY BREAKDOWN saved.
  23 unique codes evaluated
  Overall top-1: 74.0%
  Overall top-3: 96.0%

CONFIDENCE DISTRIBUTION saved.
  150 predicted-code rows (50 notes × avg 3.0 predictions each)
  Correct predictions : 53 (35.3%)

Avg confidence by correctness:
is_correct
Wrong      0.538
Correct    0.805

Avg confidence by rank:
rank
1    0.851
2    0.592
3    0.454


In [ ]:
code_accuracy = results_df.groupby('true_code').agg(
    total=('top1_correct', 'count'),
    top1_hits=('top1_correct', 'sum'),
    top3_hits=('top3_correct', 'sum')
).reset_index()
code_accuracy['top1_accuracy'] = code_accuracy['top1_hits'] / code_accuracy['total']
code_accuracy['top3_accuracy'] = code_accuracy['top3_hits'] / code_accuracy['total']
code_accuracy['icd_chapter'] = code_accuracy['true_code'].str[0]  # A=Infectious, I=Circulatory, etc.
code_accuracy.to_csv(f'{OUTPUT_PATH}/code_accuracy_breakdown.csv', index=False)

## Inspect Failures

In [ ]:
# Look at cases the model got wrong — helps you understand where to improve

wrong = results_df[~results_df['top3_correct']]
right = results_df[results_df['top1_correct']]

# build a lookup dict: code → human readable description
code_to_desc = dict(zip(guidelines_df['icd_code'], guidelines_df['description']))

print(f'Top-1 correct : {len(right)}/{len(results_df)}')
print(f'Top-3 wrong   : {len(wrong)}/{len(results_df)}')
print()

print('FAILURES (true code not in top 3):')
for _, row in wrong.head(8).iterrows():
    true_code = row['true_code']
    true_desc = code_to_desc.get(true_code, 'Description not found')

    print(f"  True code : {true_code} — {true_desc}")

    predicted = row['predicted_codes']
    if predicted:
        for code in predicted:
            desc = code_to_desc.get(code, 'Unknown / hallucinated code')
            print(f"    Predicted : {code} — {desc}")
    else:
        print(f"    Predicted : (no codes returned)")
    print()

Top-1 correct : 37/50
Top-3 wrong   : 2/50

FAILURES (true code not in top 3):
  True code : E876 — Hypokalemia
    Predicted : A419 — Sepsis, unspecified organism
    Predicted : I480 — Paroxysmal atrial fibrillation
    Predicted : N179 — Acute kidney failure, unspecified

  True code : E860 — Dehydration
    Predicted : E876 — Hypokalemia
    Predicted : I10 — Essential (primary) hypertension
    Predicted : E1165 — Type 2 diabetes mellitus with hyperglycemia



## Hallucination Evaluation
Compares RAG vs no-RAG to measure how much retrieval helps


In [ ]:
val_df_clean = val_df[
    val_df['discharge_summary'].notna() & (val_df['discharge_summary'] != '')
].copy()

def predict_codes_no_rag(discharge_summary, top_k=3):
    prompt = f"""You are an expert medical coder.
Assign ICD-10 codes to the clinical note below.

CLINICAL NOTE:
{discharge_summary[:600]}

Return EXACTLY {top_k} ICD-10 codes. Output ONLY valid JSON, nothing else.

{{"codes": [
  {{"code": "I480", "confidence": 0.95, "reason": "atrial fibrillation"}},
  {{"code": "I10",  "confidence": 0.80, "reason": "hypertension"}},
  {{"code": "E119", "confidence": 0.65, "reason": "diabetes"}}
]}}"""

    try:
        raw_text, usage = call_api_with_retry(
            messages=[
                {'role': 'system', 'content': 'You are a JSON API. Output ONLY raw JSON starting with {.'},
                {'role': 'user',   'content': prompt}
            ],
            max_tokens = 2000
        )

        clean = re.sub(r'```json|```', '', raw_text).strip()
        clean = re.sub(r',\s*([}\]])', r'\1', clean)
        last_brace = clean.rfind('}')
        if last_brace != -1:
            clean = clean[:last_brace + 1]
            clean += ']' * (clean.count('[') - clean.count(']'))
            clean += '}' * (clean.count('{') - clean.count('}'))

        try:
            data = json.loads(clean)
        except json.JSONDecodeError:
            raw_codes = re.findall(r'"code"\s*:\s*"([^"]+)"', raw_text)
            return [c.replace('.', '') for c in raw_codes[:top_k]]

        codes = data.get('codes', [])
        if codes:
            token_tracker['no_rag']['input']  += usage.prompt_tokens
            token_tracker['no_rag']['output'] += usage.completion_tokens
            token_tracker['no_rag']['calls']  += 1
            return [c['code'].replace('.', '') for c in codes[:top_k]]

    except Exception as e:
        print(f'  No-RAG failed after retries: {e}')

    return []

# --- Run comparison on same clean validation rows ---
HALLUCINATION_SAMPLES = 50

# Load all valid ICD codes from guidelines — anything outside this is a hallucination
valid_codes = set(guidelines_df['icd_code'])

print(f'Running RAG vs No-RAG comparison on {HALLUCINATION_SAMPLES} samples...')
print('✓ = Both correct | R = RAG correct | G = GPT-only correct | . = both wrong\n')

def run_halluc():
    local_rag     = []
    local_no_rag  = []

    for idx, row in val_df_clean.head(HALLUCINATION_SAMPLES).iterrows():

        note      = remove_diagnoses_section(row['discharge_summary'])
        true_code = row['primary_icd_code']

        rag_result   = predict_codes(note, vectorstore, top_k=3)
        rag_codes    = [p[0] for p in rag_result['predictions']]
        no_rag_codes = predict_codes_no_rag(note, top_k=3)

        rag_top3    = true_code in rag_codes
        no_rag_top3 = true_code in no_rag_codes

        rag_hallucinations    = [c for c in rag_codes    if c not in valid_codes]
        no_rag_hallucinations = [c for c in no_rag_codes if c not in valid_codes]

        local_rag.append({
            'true_code':        true_code,
            'predicted_codes':  rag_codes,
            'top3_correct':     rag_top3,
            'hallucinations':   rag_hallucinations,
            'n_hallucinations': len(rag_hallucinations)
        })

        local_no_rag.append({
            'true_code':        true_code,
            'predicted_codes':  no_rag_codes,
            'top3_correct':     no_rag_top3,
            'hallucinations':   no_rag_hallucinations,
            'n_hallucinations': len(no_rag_hallucinations)
        })

        if rag_top3 and not no_rag_top3:
            print('R', end='', flush=True)
        elif no_rag_top3 and not rag_top3:
            print('G', end='', flush=True)
        elif rag_top3 and no_rag_top3:
            print('✓', end='', flush=True)
        else:
            print('.', end='', flush=True)

    print('\n')
    return local_rag, local_no_rag

(rag_results, no_rag_results), carbon_halluc = run_with_carbon(
    'hallucination_eval',
    run_halluc,
    notebook='single',
    output_path=OUTPUT_PATH
)

rag_df    = pd.DataFrame(rag_results)
no_rag_df = pd.DataFrame(no_rag_results)

print('=' * 55)
print('HALLUCINATION EVALUATION RESULTS')
print('=' * 55)
print(f"{'Metric':<35} {'RAG':>8} {'No RAG':>8}")
print('-' * 55)
print(f"{'Top-3 Accuracy':<35} {rag_df['top3_correct'].mean():>8.1%} {no_rag_df['top3_correct'].mean():>8.1%}")
print(f"{'Samples with hallucinations':<35} {(rag_df['n_hallucinations'] > 0).sum():>8} {(no_rag_df['n_hallucinations'] > 0).sum():>8}")
print(f"{'Total hallucinated codes':<35} {rag_df['n_hallucinations'].sum():>8} {no_rag_df['n_hallucinations'].sum():>8}")
print(f"{'Avg hallucinations per sample':<35} {rag_df['n_hallucinations'].mean():>8.2f} {no_rag_df['n_hallucinations'].mean():>8.2f}")
print('=' * 55)

accuracy_lift = rag_df['top3_correct'].mean() - no_rag_df['top3_correct'].mean()
halluc_reduction = no_rag_df['n_hallucinations'].sum() - rag_df['n_hallucinations'].sum()

print(f'\nRAG accuracy lift:        {accuracy_lift:+.1%}')
print(f'Hallucination reduction:  {halluc_reduction} fewer hallucinated codes')

total_rag_predictions    = rag_df['predicted_codes'].apply(len).sum()
total_no_rag_predictions = no_rag_df['predicted_codes'].apply(len).sum()

print(f'\nTotal predictions evaluated:')
print(f'  RAG    : {total_rag_predictions}')
print(f'  No-RAG : {total_no_rag_predictions}')

# Save
rag_df.to_csv(f'{OUTPUT_PATH}/rag_results.csv',    index=False)
no_rag_df.to_csv(f'{OUTPUT_PATH}/no_rag_results.csv', index=False)
print('\n✓ Results saved to Drive')

Running RAG vs No-RAG comparison on 50 samples...
✓ = Both correct | R = RAG correct | G = GPT-only correct | . = both wrong

CarbonTracker: INFO - Detected CPU: Intel(R) Xeon(R) CPU @ 2.20GHz
CarbonTracker: WARNING - No matching TDP found for CPU: Intel(R) Xeon(R) CPU @ 2.20GHz. Using average TDP of 35.61W at 50% utilization as fallback.
CarbonTracker: WARNING - No API keys provided. Skipping intensity provider initialization.
CarbonTracker: WARNING - No carbon intensity provider specified. Using average carbon intensity for US: 383.55 gCO2eq/kWh.
✓✓RR✓✓R✓✓✓RRR✓  Attempt 1 failed: No JSON found. finish=length content_start=''
  Retry 1/3 after 2s...
  Attempt 2 failed: No JSON found. finish=length content_start=''
  Retry 2/3 after 4s...
✓✓✓  Attempt 1 failed: No JSON found. finish=length content_start=''
  Retry 1/3 after 2s...
  Attempt 2 failed: No JSON found. finish=length content_start=''
  Retry 2/3 after 4s...
  Attempt 3 failed: No JSON found. finish=length content_start=''
  

In [ ]:
# ── Hallucination Results Table — Power BI Export (Single Code) ──────────────
#
#
# Columns produced:
#   sample_id          — row number (1–50)
#   hadm_id            — patient encounter ID
#   true_code          — ground truth ICD-10 code
#   true_code_desc     — human-readable description of true code
#   icd_chapter        — first letter of true code (A=Infectious, I=Circulatory…)
#   mode               — "RAG" or "No-RAG"
#   predicted_codes    — pipe-separated list of predicted codes  e.g. "I10|I214|Z87"
#   n_predicted        — how many codes were returned
#   top1_correct       — 1 if rank-1 prediction == true_code, else 0
#   top3_correct       — 1 if true_code appears anywhere in predictions, else 0
#   hallucinated_codes — pipe-separated codes that don't exist in valid ICD-10 set
#   n_hallucinations   — count of hallucinated codes in this prediction
#   has_hallucination  — 1/0 flag (useful for Power BI measures)
#   hallucination_rate — n_hallucinations / n_predicted  (0.0 – 1.0)
#   rag_better         — 1 if RAG correct but No-RAG wrong (only set on RAG rows)
#   norag_better       — 1 if No-RAG correct but RAG wrong (only set on RAG rows)


# ── pull hadm_ids in order so we can join back to the patient ──────────────
hadm_ids = val_df_clean.head(HALLUCINATION_SAMPLES)['hadm_id'].tolist()

# ── build per-mode dataframes ──────────────────────────────────────────────
def build_export_df(result_list, hadm_ids, mode_label, code_to_desc):
    rows = []
    for i, (r, hid) in enumerate(zip(result_list, hadm_ids)):
        true_code  = r['true_code']
        preds      = r['predicted_codes']          # already a list
        halluc     = r['hallucinations']            # already a list
        n_pred     = len(preds)
        n_halluc   = len(halluc)

        rows.append({
            'sample_id':          i + 1,
            'hadm_id':            hid,
            'true_code':          true_code,
            'true_code_desc':     code_to_desc.get(true_code, 'Unknown'),
            'icd_chapter':        true_code[0] if true_code else '',
            'mode':               mode_label,
            'predicted_codes':    '|'.join(preds),
            'n_predicted':        n_pred,
            'top1_correct':       int(len(preds) > 0 and preds[0] == true_code),
            'top3_correct':       int(r['top3_correct']),
            'hallucinated_codes': '|'.join(halluc),
            'n_hallucinations':   n_halluc,
            'has_hallucination':  int(n_halluc > 0),
            'hallucination_rate': round(n_halluc / n_pred, 4) if n_pred > 0 else 0.0,
        })
    return pd.DataFrame(rows)


code_to_desc = dict(zip(guidelines_df['icd_code'], guidelines_df['description']))

rag_export    = build_export_df(rag_results,    hadm_ids, 'RAG',    code_to_desc)
no_rag_export = build_export_df(no_rag_results, hadm_ids, 'No-RAG', code_to_desc)

# ── add "who won" flags — only meaningful when comparing the pair ──────────
rag_correct    = rag_export.set_index('sample_id')['top3_correct']
no_rag_correct = no_rag_export.set_index('sample_id')['top3_correct']

rag_export['rag_better']   = (rag_correct.values == 1) & (no_rag_correct.values == 0)
rag_export['norag_better'] = (rag_correct.values == 0) & (no_rag_correct.values == 1)
rag_export['rag_better']   = rag_export['rag_better'].astype(int)
rag_export['norag_better'] = rag_export['norag_better'].astype(int)

# mirror columns on no_rag rows so schema stays consistent
no_rag_export['rag_better']   = rag_export['rag_better'].values
no_rag_export['norag_better'] = rag_export['norag_better'].values

# ── combine and save ───────────────────────────────────────────────────────
hallucination_export = pd.concat([rag_export, no_rag_export], ignore_index=True)
hallucination_export = hallucination_export.sort_values(['sample_id', 'mode']).reset_index(drop=True)

out_path = f'{OUTPUT_PATH}/hallucination_results_single.csv'
hallucination_export.to_csv(out_path, index=False)

# ── quick sanity summary ───────────────────────────────────────────────────
print('=' * 60)
print('HALLUCINATION EXPORT — SINGLE CODE')
print('=' * 60)
print(f'Rows written : {len(hallucination_export)}  ({HALLUCINATION_SAMPLES} samples × 2 modes)')
print(f'Saved to     : {out_path}')
print()

summary = hallucination_export.groupby('mode').agg(
    top3_accuracy   = ('top3_correct',    'mean'),
    halluc_samples  = ('has_hallucination','sum'),
    total_halluc    = ('n_hallucinations', 'sum'),
    avg_halluc      = ('n_hallucinations', 'mean'),
).round(3)
print(summary.to_string())
print()
print('Preview (first 4 rows):')
print(hallucination_export[['sample_id','mode','true_code','top3_correct',
                             'n_hallucinations','hallucinated_codes']].head(4).to_string(index=False))

HALLUCINATION EXPORT — SINGLE CODE
Rows written : 100  (50 samples × 2 modes)
Saved to     : /content/drive/MyDrive/mimic4_medical_coding/hallucination_results_single.csv

        top3_accuracy  halluc_samples  total_halluc  avg_halluc
mode                                                           
No-RAG           0.54               2             2        0.04
RAG              0.98               0             0        0.00

Preview (first 4 rows):
 sample_id   mode true_code  top3_correct  n_hallucinations hallucinated_codes
         1 No-RAG      E119             1                 0                   
         1    RAG      E119             1                 0                   
         2 No-RAG     G4733             1                 0                   
         2    RAG     G4733             1                 0                   


## Get final API cost per run

In [ ]:
def calculate_api_cost():
    PRICE_INPUT  = 0.50 / 1000000   # $0.50 per million input tokens
    PRICE_OUTPUT = 2.00 / 1000000   # $2.00 per million output tokens

    rows = []
    for mode, t in token_tracker.items():
        if t['calls'] == 0:
            continue
        in_cost  = t['input']  * PRICE_INPUT
        out_cost = t['output'] * PRICE_OUTPUT
        total    = in_cost + out_cost
        rows.append({
            'mode':               'RAG' if mode == 'rag' else 'No-RAG',
            'calls':              t['calls'],
            'input_tokens':       t['input'],
            'output_tokens':      t['output'],
            'avg_input_tokens':   round(t['input']  / t['calls']),
            'avg_output_tokens':  round(t['output'] / t['calls']),
            'input_cost':         round(in_cost,  5),
            'output_cost':        round(out_cost, 5),
            'total_cost':         round(total,    5),
            'cost_per_call':      round(total / t['calls'], 6)
        })

    cost_df = pd.DataFrame(rows)

    print('=' * 55)
    print('ACTUAL API COST — THIS RUN')
    print('=' * 55)
    for _, r in cost_df.iterrows():
        print(f"\n  [{r['mode']}]")
        print(f"    Calls              : {r['calls']}")
        print(f"    Avg input tokens   : {r['avg_input_tokens']:,}  ← prompt size")
        print(f"    Avg output tokens  : {r['avg_output_tokens']:,}  ← response size")
        print(f"    Total cost         : ${r['total_cost']:.5f}")
        print(f"    Cost per call      : ${r['cost_per_call']:.6f}")
    print('=' * 55)

    if len(cost_df) == 2:
        rag_cost    = cost_df[cost_df['mode'] == 'RAG']['total_cost'].values[0]
        no_rag_cost = cost_df[cost_df['mode'] == 'No-RAG']['total_cost'].values[0]
        overhead    = rag_cost - no_rag_cost
        print(f"\n  RAG overhead vs No-RAG : ${overhead:.5f} (+{overhead/no_rag_cost*100:.1f}%)")
        print(f"  (larger prompts due to retrieved context)")

    # save for dashboard
    cost_df.to_csv(f'{OUTPUT_PATH}/api_cost_results.csv', index=False)
    print(f"\n  ✓ Saved to api_cost_results.csv")
    return cost_df

cost_df = calculate_api_cost()

cost_df['notebook'] = 'single'
cost_df.to_csv(f'{OUTPUT_PATH}/api_cost.csv', index=False)

ACTUAL API COST — THIS RUN

  [RAG]
    Calls              : 104
    Avg input tokens   : 439  ← prompt size
    Avg output tokens  : 1,033  ← response size
    Total cost         : $0.23768
    Cost per call      : $0.002285

  [No-RAG]
    Calls              : 50
    Avg input tokens   : 291  ← prompt size
    Avg output tokens  : 1,088  ← response size
    Total cost         : $0.11604
    Cost per call      : $0.002321

  RAG overhead vs No-RAG : $0.12164 (+104.8%)
  (larger prompts due to retrieved context)

  ✓ Saved to api_cost_results.csv


## Sample Synthetic Clinical Note

In [ ]:
# ── Example synthetic clinical note ──────────────────────
print('=' * 60)
print('EXAMPLE SYNTHETIC CLINICAL NOTE')
print('=' * 60)
example_note = synthetic_df.iloc[0]
print(f"ICD-10 Code : {example_note['primary_icd_code']}")
print(f"Description : {example_note['primary_icd_description']}")
print(f"\nNote:\n{example_note['discharge_summary']}")

# ── Example ICD-10 guideline ─────────────────────────────
print('\n' + '=' * 60)
print('EXAMPLE ICD-10 GUIDELINE')
print('=' * 60)
example_guideline = guidelines_df[guidelines_df['icd_code'] == 'I214'].iloc[0]
print(f"Code        : {example_guideline['icd_code']}")
print(f"Description : {example_guideline['description']}")

EXAMPLE SYNTHETIC CLINICAL NOTE
ICD-10 Code : I214
Description : Non-ST elevation (NSTEMI) myocardial infarction

Note:
Chief complaint: Substernal chest pain. History: 62yo M with sudden onset of severe substernal chest pain radiating to left arm, associated with diaphoresis and nausea. Pain began 2 hours prior to arrival and is unrelieved by sublingual nitroglycerin. PMHx: HTN, dyslipidemia. No prior cardiac history. Physical: BP 145/90, HR 110, RR 18, temp 37.1°C. Lung fields clear. S1S2 with regular rate and rhythm. No murmurs, rubs, or gallops. No JVD or lower extremity edema. Hospital course: Initially taken to cath lab showing 100% occlusion of LAD. Successful PCI with stent placement. Peak troponin 23 ng/mL. Discharged on ASA, clopidogrel, lisinopril, atorvastatin, and metoprolol. Discharge plan: Outpatient cardiac rehab follow-up in 1 week. Discharged on home O2 nocturnally.

EXAMPLE ICD-10 GUIDELINE
Code        : I214
Description : Non-ST elevation (NSTEMI) myocardial infarct

##Consolidate CarbonTracker data

In [ ]:
def build_carbon_summary(output_path, notebook, results_df, carbon_build,
                         carbon_eval, carbon_halluc, cost_df, eval_samples):
    """
    Combines all carbon tracking stages + accuracy + cost into one Power BI CSV.

    Columns:
      notebook          — 'single' or 'multi'
      stage             — 'vector_store_build' | 'rag_eval' | 'hallucination_eval'
      co2_grams         — total CO2 for that stage
      energy_kwh        — total energy for that stage
      n_predictions     — how many predictions were made (0 for build stage)
      co2_per_pred      — co2_grams / n_predictions (null for build)
      top1_accuracy     — from results_df (null for non-eval stages)
      top3_accuracy     — from results_df (null for non-eval stages)
      rag_total_cost    — from cost_df
      norag_total_cost  — from cost_df
      rag_cost_per_call — from cost_df
      timestamp         — when this run happened
    """
    timestamp = pd.Timestamp.now().isoformat()

    # accuracy metrics from results_df
    top1_acc = round(results_df['top1_correct'].mean(), 4)
    top3_acc = round(results_df['top3_correct'].mean(), 4)

    # cost from cost_df
    rag_row    = cost_df[cost_df['mode'] == 'RAG'].iloc[0]    if 'RAG'    in cost_df['mode'].values else None
    norag_row  = cost_df[cost_df['mode'] == 'No-RAG'].iloc[0] if 'No-RAG' in cost_df['mode'].values else None

    def stage_row(carbon, n_preds=0, include_accuracy=False):
        co2 = carbon.get('co2_grams')
        return {
            'notebook':          notebook,
            'stage':             carbon['stage'],
            'co2_grams':         co2,
            'energy_kwh':        carbon.get('energy_kwh'),
            'n_predictions':     n_preds,
            'co2_per_pred':      round(co2 / n_preds, 8) if (co2 and n_preds > 0) else None,
            'top1_accuracy':     top1_acc if include_accuracy else None,
            'top3_accuracy':     top3_acc if include_accuracy else None,
            'rag_total_cost':    rag_row['total_cost']      if rag_row    is not None else None,
            'norag_total_cost':  norag_row['total_cost']    if norag_row  is not None else None,
            'rag_cost_per_call': rag_row['cost_per_call']   if rag_row    is not None else None,
            'timestamp':         timestamp,
        }

    rows = [
        stage_row(carbon_build,  n_preds=0,            include_accuracy=False),
        stage_row(carbon_eval,   n_preds=eval_samples,  include_accuracy=True),
        stage_row(carbon_halluc, n_preds=eval_samples,  include_accuracy=False),
    ]

    summary_df = pd.DataFrame(rows)
    out_path = f'{output_path}/carbon_summary_{notebook}.csv'
    summary_df.to_csv(out_path, index=False)

    print('=' * 55)
    print(f'CARBON SUMMARY — {notebook.upper()} CODE')
    print('=' * 55)
    print(summary_df[['stage', 'co2_grams', 'energy_kwh', 'n_predictions', 'co2_per_pred']].to_string(index=False))
    print(f'\n✓ Saved to {out_path}')

    return summary_df

In [ ]:
carbon_summary = build_carbon_summary(
output_path    = OUTPUT_PATH,
notebook       = 'single',
results_df     = results_df,
carbon_build   = carbon_build,
carbon_eval    = carbon_eval,
carbon_halluc  = carbon_halluc,
cost_df        = cost_df,
eval_samples   = EVAL_SAMPLES)

CARBON SUMMARY — SINGLE CODE
             stage  co2_grams  energy_kwh  n_predictions  co2_per_pred
vector_store_build   0.698128    0.001820              0           NaN
          rag_eval   2.773540    0.007231             50      0.055471
hallucination_eval   3.585493    0.009348             50      0.071710

✓ Saved to /content/drive/MyDrive/mimic4_medical_coding/carbon_summary_single.csv
